In [1]:
import numpy as np 
import pandas as pd 
import  data_clean_utils

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, PowerTransformer, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Configure Plotly for Jupyter rendering
import plotly.io as pio
pio.renderers.default = 'notebook'

In [2]:
!pip install mlflow dagshub 

In [3]:
import dagshub
dagshub.auth.add_app_token("1430185965d5cc1c10ff15d59e0405f69b16adca")

In [4]:
dagshub.init(
    repo_owner='anni0955',
    repo_name='delivery-time-prediction',
    mlflow=True
)

Accessing as anni0955

Initialized MLflow to track repo "anni0955/delivery-time-prediction"

Repository anni0955/delivery-time-prediction initialized!

In [5]:
import mlflow 

In [6]:
mlflow.set_tracking_uri('https://dagshub.com/anni0955/delivery-time-prediction.mlflow')

In [7]:
mlflow.set_experiment('Exp 7 - Final Estimator')

2026/03/18 04:58:42 INFO mlflow.tracking.fluent: Experiment with name 'Exp 7 - Final Estimator' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/93ceafb4a70644539342e344ecb6b480', creation_time=1773809922458, experiment_id='8', last_update_time=1773809922458, lifecycle_stage='active', name='Exp 7 - Final Estimator', tags={}, workspace='default'>

In [8]:
from sklearn import set_config
set_config(transform_output='pandas')

## Load data

In [9]:
df = pd.read_csv('train.csv')

In [10]:
data_clean_utils.perform_data_cleanining(df)

In [11]:
df = pd.read_csv('cleaned-data.csv')
df

,rider_id,rider_age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,INDO,19,3,saturday,1,15.0,11.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,BANG,25,3,friday,0,5.0,19.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,BANG,19,3,saturday,1,15.0,8.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,COIMB,5,4,tuesday,0,10.0,18.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,CHEN,26,3,saturday,1,15.0,13.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,JAPRES04DEL01,30.0,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24,windy,high,...,JAP,24,3,thursday,0,10.0,11.0,morning,1.489846,short
45498,AGRRES16DEL01,21.0,4.6,NaN,NaN,NaN,NaN,2022-02-16,windy,jam,...,AGR,16,2,wednesday,0,15.0,19.0,evening,NaN,NaN
45499,CHENRES08DEL03,30.0,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11,cloudy,low,...,CHEN,11,3,friday,0,15.0,23.0,night,4.657195,short
45500,COIMBRES11DEL01,20.0,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07,cloudy,high,...,COIMB,7,3,monday,0,5.0,13.0,afternoon,6.232393,medium


In [12]:
cols_to_drop = [
    'rider_id',
    'restaurant_latitude',
    'restaurant_longitude',
    'delivery_latitude',
    'delivery_longitude',
    'order_date',
    'city_name',
    'order_month',
    'order_day_of_week',
    'order_time_hour',
    'order_day'
]

df.drop(columns=cols_to_drop, inplace=True)
df

,rider_age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [13]:
df.isna().sum()

,0
rider_age,1854
ratings,1908
weather,525
traffic,510
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,993
festival,228
city_type,1198


In [14]:
temp_df = df.copy().dropna()
x = temp_df.drop(columns=['time_taken'])
y = temp_df['time_taken']

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.2, random_state=42)

In [16]:
x_train.isna().sum()

,0
rider_age,0
ratings,0
weather,0
traffic,0
vehicle_condition,0
type_of_order,0
type_of_vehicle,0
multiple_deliveries,0
festival,0
city_type,0


In [17]:
pt = PowerTransformer()
y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

## preprocessing pipeline 

In [18]:
x_train.columns

Index(['rider_age', 'ratings', 'weather', 'traffic', 'vehicle_condition',
       'type_of_order', 'type_of_vehicle', 'multiple_deliveries', 'festival',
       'city_type', 'is_weekend', 'pickup_time_minutes', 'order_time_of_day',
       'distance', 'distance_type'],
      dtype='object')

In [19]:
num_cols = ['rider_age', 'ratings', 
            'pickup_time_minutes', 'distance']

nominal_cat_cols = ['weather', 'type_of_vehicle', 
                    'festival', 'city_type',
                    'is_weekend', 'order_time_of_day', 
                    'type_of_order']

ordinal_cat_cols = ['traffic', 'distance_type']

traffic_order = ['low', 'medium', 'high', 'jam']
distance_type_order = ['short', 'medium', 'long', 
                       'very_long']

In [20]:
preprocessor = ColumnTransformer(transformers=[
    ('scale', MinMaxScaler(), num_cols),
    ('nominal_encode', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), nominal_cat_cols),
    ('ordinal_encode', OrdinalEncoder(categories=[traffic_order, distance_type_order], encoded_missing_value=-999, handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cat_cols),
], remainder='passthrough', n_jobs=-1, verbose_feature_names_out=False, force_int_remainder_cols=False)

preprocessor

ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                  remainder='passthrough',
                  transformers=[('scale', MinMaxScaler(),
                                 ['rider_age', 'ratings', 'pickup_time_minutes',
                                  'distance']),
                                ('nominal_encode',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 ['weather', 'type_of_vehicle', 'festival',
                                  'city_type', 'is_weekend',
                                  'order_time_of_day', 'type_of_order']),
                                ('ordinal_encode',
                                 OrdinalEncoder(categories=[['low', 'medium',
                                                             'high', 'jam'],
                                                            ['short', 'medium',
                                                             'long',
                                                             'very_long']],
                                                encoded_missing_value=-999,
                                                handle_unknown='use_encoded_value',
                                                unknown_value=-1),
                                 ['traffic', 'distance_type'])],
                  verbose_feature_names_out=False)

In [21]:
preprocessing_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

preprocessing_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(force_int_remainder_cols=False, n_jobs=-1,
                                   remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  ['rider_age', 'ratings',
                                                   'pickup_time_minutes',
                                                   'distance']),
                                                 ('nominal_encode',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['weather', 'type_of_vehicle',
                                                   'festival', 'city_type',
                                                   'is_weekend',
                                                   'order_time_of_day',
                                                   'type_of_order']),
                                                 ('ordinal_encode',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'medium',
                                                                              'high',
                                                                              'jam'],
                                                                             ['short',
                                                                              'medium',
                                                                              'long',
                                                                              'very_long']],
                                                                 encoded_missing_value=-999,
                                                                 handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['traffic',
                                                   'distance_type'])],
                                   verbose_feature_names_out=False))])

In [22]:
x_train_trans = preprocessing_pipeline.fit_transform(x_train)
x_test_trans = preprocessing_pipeline.transform(x_test)

x_train_trans

,rider_age,ratings,pickup_time_minutes,distance,weather_fog,weather_sandstorms,weather_stormy,weather_sunny,weather_windy,type_of_vehicle_motorcycle,...,order_time_of_day_evening,order_time_of_day_morning,order_time_of_day_night,type_of_order_drinks,type_of_order_meal,type_of_order_snack,traffic,distance_type,vehicle_condition,multiple_deliveries
8708,0.473684,0.56,1.0,0.404165,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,1.0,0.0,0.0,3.0,1.0,0,2.0
25198,1.000000,0.76,0.0,0.154044,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0,1.0
34049,0.473684,0.80,0.5,0.002461,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,2.0,0.0,1,0.0
25987,1.000000,0.92,1.0,0.460411,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,2.0,0,1.0
37121,0.526316,0.76,0.5,0.243676,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20239,0.578947,0.92,0.5,0.451895,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,3.0,2.0,0,0.0
7590,0.052632,1.00,1.0,0.612270,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,1.0,2.0,1,1.0
13610,0.526316,0.92,0.0,0.322877,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1,0.0
1045,0.947368,0.96,0.5,0.004486,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1.0


In [23]:
%pip install optuna 

In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor 

import optuna
from sklearn.metrics import r2_score, mean_absolute_error

In [25]:
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor 
from sklearn.ensemble import StackingRegressor

In [26]:
best_rf_params = {
    'n_estimators': 297,
    'max_depth': 14,
    'max_features': None,
    'min_samples_split': 5,
    'min_samples_leaf': 3,
    'max_samples': 0.6884129569843086
}

best_lgbm_params = {
    'n_estimators': 142,
    'max_depth': 19,
    'learning_rate': 0.4634183035475242,
    'subsample': 0.7628386779555664,
    'min_child_weight': 14,
    'min_split_gain': 1.0050226994104905,
    'reg_lambda': 7.743851432894324
}

best_rf = RandomForestRegressor(**best_rf_params)
best_lgbm = LGBMRegressor(**best_lgbm_params)

lr = LinearRegression()

In [28]:
stacking_reg = StackingRegressor(estimators=[('rf', best_rf), ('lgbm', best_lgbm)],
                                 final_estimator=lr, cv=5, 
                                 n_jobs=-1)

model = TransformedTargetRegressor(regressor=stacking_reg, transformer=pt)

model.fit(x_train_trans, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning:

X has feature names, but PowerTransformer was fitted without feature names



TransformedTargetRegressor(regressor=StackingRegressor(cv=5,
                                                       estimators=[('rf',
                                                                    RandomForestRegressor(max_depth=14,
                                                                                          max_features=None,
                                                                                          max_samples=0.6884129569843086,
                                                                                          min_samples_leaf=3,
                                                                                          min_samples_split=5,
                                                                                          n_estimators=297)),
                                                                   ('lgbm',
                                                                    LGBMRegressor(learning_rate=0.4634183035475242,
                                                                                  max_depth=19,
                                                                                  min_child_weight=14,
                                                                                  min_split_gain=1.0050226994104905,
                                                                                  n_estimators=142,
                                                                                  reg_lambda=7.743851432894324,
                                                                                  subsample=0.7628386779555664))],
                                                       final_estimator=LinearRegression(),
                                                       n_jobs=-1),
                           transformer=PowerTransformer())

In [29]:
y_train_pred = model.predict(x_train_trans)
y_test_pred = model.predict(x_test_trans)

train_r2_score = r2_score(y_train, y_train_pred)
test_r2_score = r2_score(y_test, y_test_pred)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning:

X has feature names, but LinearRegression was fitted without feature names

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning:

X has feature names, but LinearRegression was fitted without feature names



In [30]:
cv_scores = cross_val_score(model, x_train_trans, y_train, cv=3, 
                            scoring='neg_mean_absolute_error', 
                            n_jobs=-1)

-cv_scores

array([3.08766042, 3.10981033, 3.08181446])

In [31]:
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

In [36]:
with mlflow.start_run():
    mlflow.set_tag('model', 'stacking_regressor')
    mlflow.log_params(stacking_reg.get_params())

    mlflow.log_metric('train_mae', train_mae)
    mlflow.log_metric('test_mae', test_mae)
    mlflow.log_metric('train_r2_score', train_r2_score)
    mlflow.log_metric('test_r2_score', test_r2_score)

    mlflow.sklearn.log_model(stacking_reg, 'model')


2026/03/18 06:06:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/18 06:06:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run shivering-panda-614 at: https://dagshub.com/anni0955/delivery-time-prediction.mlflow/#/experiments/8/runs/facf4b09be5a4ddfab257a5372d10ddd
🧪 View experiment at: https://dagshub.com/anni0955/delivery-time-prediction.mlflow/#/experiments/8
